## RAG Local Chatbot

### Expert Question Answerer for InsureLLM
### Compliance-Safe: All processing runs locally via Ollama

LangChain 1.0 implementation of a RAG pipeline using local Ollama.

Using the VectorStore we created with HuggingFace `all-MiniLM-L6-v2` embeddings

In [2]:
from dotenv import load_dotenv
from langchain_openai import ChatOpenAI
from langchain_chroma import Chroma
from langchain_core.messages import SystemMessage, HumanMessage
from langchain_huggingface import HuggingFaceEmbeddings
import gradio as gr
import os

In [3]:
# COMPLIANCE: Local-only configuration
# All data processing happens on this machine via Ollama
# No data is sent to external APIs or services

MODEL = "llama3.2"   # or "llama3.2:1b" on a smaller machine
OLLAMA_BASE_URL = "http://localhost:11434/v1"
DB_NAME = "vector_db"
load_dotenv(override=True)

True

### Connect to Chroma; use HuggingFace all-MiniLM-L6-v2 (LOCAL)

In [4]:
embeddings = HuggingFaceEmbeddings(model_name="all-MiniLM-L6-v2")
vectorstore = Chroma(persist_directory=DB_NAME, embedding_function=embeddings)

### Set up the 2 key LangChain objects: retriever and llm

#### A sidebar on "temperature":
- Controls how diverse the output is
- A temperature of 0 means that the output should be predictable
- Higher temperature for more variety in answers

Some people describe temperature as being like 'creativity' but that's not quite right
- It actually controls which tokens get selected during inference
- temperature=0 means: always select the token with highest probability
- temperature=1 usually means: a token with 10% probability should be picked 10% of the time

Note: a temperature of 0 doesn't mean outputs will always be reproducible. You also need to set a random seed.

In [5]:
retriever = vectorstore.as_retriever()

# COMPLIANCE: Using local Ollama, not cloud-based OpenAI
llm = ChatOpenAI(
    temperature=0,
    model=MODEL,
    base_url=OLLAMA_BASE_URL,
    api_key="ollama",  # dummy key; Ollama doesn't need a real one
)

### These LangChain objects implement the method `invoke()`

In [6]:
retriever.invoke("Who is Avery?")

[Document(id='c7d9290d-d184-4d25-9914-62e015410fe0', metadata={'doc_type': 'employees', 'source': 'knowledge-base/employees/Avery Lancaster.md'}, page_content="## Other HR Notes\n- **Professional Development**: Avery has actively participated in leadership training programs and industry conferences, representing Insurellm and fostering partnerships.  \n- **Diversity & Inclusion Initiatives**: Avery has championed a commitment to diversity in hiring practices, seeing visible improvements in team representation since 2021.  \n- **Work-Life Balance**: Feedback revealed concerns regarding work-life balance, which Avery has approached by implementing flexible working conditions and ensuring regular check-ins with the team.\n- **Community Engagement**: Avery led community outreach efforts, focusing on financial literacy programs, particularly aimed at underserved populations, improving Insurellm's corporate social responsibility image.  \n\nAvery Lancaster has demonstrated resilience and ada

In [7]:
llm.invoke("Who is Avery?")

AIMessage(content='There are several notable individuals named Avery, so it\'s possible that you\'re referring to one of the following:\n\n1. Avery Brooks: An American actor known for his roles in TV shows such as "Star Trek: Deep Space Nine" and "Designated Survivor".\n2. Avery Whitted: An American professional poker player who has won several tournaments, including the World Series of Poker (WSOP) Main Event.\n3. Avery Wilkerson: An American football linebacker who currently plays for the New York Giants in the National Football League (NFL).\n4. Avery Jones: A British singer-songwriter and musician known for his soulful voice and genre-bending sound.\n\nIf none of these individuals match the Avery you\'re thinking of, please provide more context or information about the Avery you\'re interested in, and I\'ll do my best to help!', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 171, 'prompt_tokens': 29, 'total_tokens': 200, 'completion_tok

## Time to put this together!

In [8]:
SYSTEM_PROMPT_TEMPLATE = """
You are a knowledgeable, friendly assistant representing the company Insurellm.
You are chatting with a user about Insurellm.
If relevant, use the given context to answer any question.
If you don't know the answer, say so.
Context:
{context}
"""

In [9]:
def answer_question(question: str, history):
    docs = retriever.invoke(question)
    context = "\n\n".join(doc.page_content for doc in docs)
    system_prompt = SYSTEM_PROMPT_TEMPLATE.format(context=context)
    response = llm.invoke([SystemMessage(content=system_prompt), HumanMessage(content=question)])
    return response.content

In [10]:
answer_question("Who is Avery Lancaster?", [])

"Avery Lancaster is the Co-Founder & Chief Executive Officer (CEO) of Insurellm. She has been instrumental in shaping the company's success and growth in the insurance technology landscape. With her innovative leadership strategies, risk management expertise, and commitment to diversity and inclusion, Avery has positioned Insurellm as a leading player in the industry.\n\nAvery has a strong background in product development and management, having worked as a Senior Product Manager at Innovate Insurance Solutions before co-founding Insurellm. Her experience and expertise have been instrumental in driving the company's success, including the launch of new products that have significantly increased market share.\n\nUnder Avery's leadership, Insurellm has received recognition for its innovative approaches to personalized insurance solutions, and she is now recognized as a leading voice in Insurance Tech innovation."

## Launch the Chatbot!

All conversation data stays local to this machine.

In [11]:
gr.ChatInterface(answer_question).launch()

/Users/priyesh/projects/llm_engineering/.venv/lib/python3.12/site-packages/gradio/chat_interface.py:347: UserWarning: The 'tuples' format for chatbot messages is deprecated and will be removed in a future version of Gradio. Please set type='messages' instead, which uses openai-style 'role' and 'content' keys.
  self.chatbot = Chatbot(


* Running on local URL:  http://127.0.0.1:7860
* To create a public link, set `share=True` in `launch()`.
